In [2]:
import os
import pandas as pd
import numpy as np

data = []
labels = []

base_path = "dataset"

# label mapping
label_map = {
    "m": 0,
    "nomotion": 1,
    "o": 2,
    "z": 3
}

for folder in os.listdir(base_path):
    folder_path = os.path.join(base_path, folder)

    if folder in label_map:
        for file in os.listdir(folder_path):
            if file.endswith(".csv"):
                file_path = os.path.join(folder_path, file)

                df = pd.read_csv(file_path)

                # assuming columns: ax, ay, az, gx, gy, gz
                values = df[['ax','ay','az','gx','gy','gz']].values

                data.append(values)
                labels.append(label_map[folder])

X = np.array(data)
y = np.array(labels)

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (240,) + inhomogeneous part.

In [ ]:
import os
import pandas as pd
import numpy as np

window_size = 50
step = 25

data = []
labels = []

base_path = "dataset"

label_map = {
    "m": 0,
    "nomotion": 1,
    "o": 2,
    "z": 3
}

for folder in os.listdir(base_path):
    folder_path = os.path.join(base_path, folder)

    if folder in label_map:
        for file in os.listdir(folder_path):
            if file.endswith(".csv"):
                file_path = os.path.join(folder_path, file)

                df = pd.read_csv(file_path)
                values = df[['ax','ay','az','gx','gy','gz']].values

                # Apply sliding window
                for i in range(0, len(values) - window_size, step):
                    window = values[i:i+window_size]
                    data.append(window)
                    labels.append(label_map[folder])

X = np.array(data)
y = np.array(labels)

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

Shape of X: (725, 50, 6)
Shape of y: (725,)


In [4]:
from sklearn.preprocessing import StandardScaler

# reshape for scaling
nsamples, timesteps, nfeatures = X.shape
X_reshaped = X.reshape(-1, nfeatures)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_reshaped)

# reshape back
X = X_scaled.reshape(nsamples, timesteps, nfeatures)

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [6]:
import tensorflow as tf
from tensorflow.keras import layers, models

model = models.Sequential([
    
    layers.Conv1D(16, kernel_size=3, activation='relu', input_shape=(50,6)),
    layers.MaxPooling1D(2),

    layers.Conv1D(32, kernel_size=3, activation='relu'),
    layers.MaxPooling1D(2),

    layers.Flatten(),

    layers.Dense(32, activation='relu'),
    layers.Dense(4, activation='softmax')   # 4 classes
])

model.summary()

c:\Users\Ariba\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 48, 16)         │           304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 24, 16)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 22, 32)         │         1,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 11, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 352)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │        11,296 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,300 (51.95 KB)

 Trainable params: 13,300 (51.95 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [8]:
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=16,
    validation_data=(X_test, y_test)
)

Epoch 1/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.5310 - loss: 1.1141 - val_accuracy: 0.8069 - val_loss: 0.8309
Epoch 2/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7983 - loss: 0.6534 - val_accuracy: 0.8759 - val_loss: 0.4919
Epoch 3/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8672 - loss: 0.3908 - val_accuracy: 0.9241 - val_loss: 0.3100
Epoch 4/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9276 - loss: 0.2463 - val_accuracy: 0.9517 - val_loss: 0.2225
Epoch 5/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9362 - loss: 0.1864 - val_accuracy: 0.9448 - val_loss: 0.1936
Epoch 6/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9603 - loss: 0.1309 - val_accuracy: 0.9586 - val_loss: 0.1338
Epoch 7/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9707 - loss: 0.0932 - val_accuracy: 0.9793 - val_loss: 0.1027
Epoch 8/20
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9810 - loss: 0.0704 - val_accuracy: 0.9862 - val_loss:

In [9]:
loss, acc = model.evaluate(X_test, y_test)
print("Test Accuracy:", acc)

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9793 - loss: 0.0443 
Test Accuracy: 0.9793103337287903


In [10]:
import json

scaler_params = {
    "mean": scaler.mean_.tolist(),
    "std": scaler.scale_.tolist()
}

with open("scaler.json", "w") as f:
    json.dump(scaler_params, f)

print("Scaler saved!")

Scaler saved!


In [11]:
import tensorflow as tf

# Convert model
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save model
with open("model.tflite", "wb") as f:
    f.write(tflite_model)

print("Model converted!")

INFO:tensorflow:Assets written to: C:\Users\Ariba\AppData\Local\Temp\tmplpdzdo3e\assets


INFO:tensorflow:Assets written to: C:\Users\Ariba\AppData\Local\Temp\tmplpdzdo3e\assets


Saved artifact at 'C:\Users\Ariba\AppData\Local\Temp\tmplpdzdo3e'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 50, 6), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  1720055586384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1720058554768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1720058553616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1720058552848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1720058554192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1720058555344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1720058555152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1720058554384: TensorSpec(shape=(), dtype=tf.resource, name=None)
Model converted!


In [12]:
def representative_data_gen():
    for i in range(100):
        yield [X_train[i:i+1].astype("float32")]

converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen

converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_quant_model = converter.convert()

with open("model_quant.tflite", "wb") as f:
    f.write(tflite_quant_model)

print("Quantized model ready!")

INFO:tensorflow:Assets written to: C:\Users\Ariba\AppData\Local\Temp\tmpgooqc9to\assets


INFO:tensorflow:Assets written to: C:\Users\Ariba\AppData\Local\Temp\tmpgooqc9to\assets


Saved artifact at 'C:\Users\Ariba\AppData\Local\Temp\tmpgooqc9to'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 50, 6), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  1720055586384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1720058554768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1720058553616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1720058552848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1720058554192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1720058555344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1720058555152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1720058554384: TensorSpec(shape=(), dtype=tf.resource, name=None)


c:\Users\Ariba\AppData\Local\Programs\Python\Python311\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Quantized model ready!
